In [2]:
import os
import shutil

JSON_PATH = "/Volumes/Extreme SSD/json_juli_1c_3c/extracted_data_3c.json"
BACKUP_PATH = JSON_PATH + ".backup"
FIXED_PATH = JSON_PATH + ".fixed"

def fix_nan_in_json(input_file, output_file):
    """
    Baca file JSON baris per baris, ganti NaN/Infinity dengan null.
    """
    print(f"🔧 Memperbaiki file: {input_file}")
    print(f"   Output sementara: {output_file}")

    with open(input_file, 'r', encoding='utf-8') as fin, \
         open(output_file, 'w', encoding='utf-8') as fout:
        for line_num, line in enumerate(fin, 1):
            # Ganti literal NaN dan Infinity (case sensitive)
            # Hati-hati: jangan sampai mengganti bagian string yang mengandung "NaN"
            # Sebaiknya gunakan regex, tapi replace sederhana cukup untuk kasus ini
            line = line.replace('NaN', 'null')
            line = line.replace('Infinity', 'null')
            line = line.replace('-Infinity', 'null')
            fout.write(line)
            if line_num % 10000 == 0:
                print(f"   Diproses {line_num} baris...")

    print("✅ Perbaikan selesai.")

# Buat backup file asli (untuk jaga-jaga)
if not os.path.exists(BACKUP_PATH):
    print("📦 Membuat backup file asli...")
    shutil.copy2(JSON_PATH, BACKUP_PATH)

# Perbaiki ke file sementara
fix_nan_in_json(JSON_PATH, FIXED_PATH)

# Ganti file asli dengan yang sudah diperbaiki
os.replace(FIXED_PATH, JSON_PATH)
print(f"✅ File asli telah diperbarui. Backup tersimpan di: {BACKUP_PATH}")

📦 Membuat backup file asli...
🔧 Memperbaiki file: /Volumes/Extreme SSD/json_juli_1c_3c/extracted_data_3c.json
   Output sementara: /Volumes/Extreme SSD/json_juli_1c_3c/extracted_data_3c.json.fixed
   Diproses 10000 baris...
   Diproses 20000 baris...
   Diproses 30000 baris...
   Diproses 40000 baris...
   Diproses 50000 baris...
   Diproses 60000 baris...
   Diproses 70000 baris...
   Diproses 80000 baris...
   Diproses 90000 baris...
   Diproses 100000 baris...
   Diproses 110000 baris...
   Diproses 120000 baris...
   Diproses 130000 baris...
   Diproses 140000 baris...
   Diproses 150000 baris...
   Diproses 160000 baris...
   Diproses 170000 baris...
   Diproses 180000 baris...
   Diproses 190000 baris...
   Diproses 200000 baris...
   Diproses 210000 baris...
   Diproses 220000 baris...
   Diproses 230000 baris...
   Diproses 240000 baris...
   Diproses 250000 baris...
   Diproses 260000 baris...
   Diproses 270000 baris...
   Diproses 280000 baris...
   Diproses 290000 baris...


In [6]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
CEK FILE JSON 3C (Menggunakan ijson untuk file besar)
"""

import ijson
import os
import json
from collections import defaultdict

JSON_PATH = "/Volumes/Extreme SSD/json_juli_1c_3c/extracted_data_3c.json"

def inspect_json_streaming(file_path):
    print("="*70)
    print("🔍 MEMERIKSA JSON 3C (Streaming mode)")
    print("="*70)
    print(f"📂 File: {file_path}")
    
    # Cek ukuran file
    file_size = os.path.getsize(file_path) / (1024**3)
    print(f"📦 Ukuran file: {file_size:.2f} GB")
    
    # Inisialisasi counter
    total_entries = 0
    sample_entries = []
    keys_found = set()
    metadata_keys = set()
    
    # Variabel untuk statistik
    z_lengths = []
    n_lengths = []
    e_lengths = []
    has_all_components = 0
    has_z_only = 0
    
    print("\n⏳ Membaca file secara streaming... (mungkin butuh beberapa detik)")
    
    with open(file_path, 'rb') as f:
        # Parse streaming
        parser = ijson.parse(f)
        
        # State untuk tracking posisi
        current_key = None
        current_event = {}
        inside_metadata = False
        
        for prefix, event, value in parser:
            # Deteksi awal objek event (top-level key)
            if event == 'map_key' and prefix == '':
                current_key = value
                current_event = {}
                total_entries += 1
                inside_metadata = False
            
            # Jika prefix adalah metadata
            if prefix.endswith('.metadata'):
                inside_metadata = True
            
            # Rekam data untuk sampel (hanya 3 event pertama)
            if total_entries <= 3:
                if event in ['number', 'string', 'boolean']:
                    # Simpan nilai untuk sampel
                    if prefix.endswith('.Z') and not prefix.endswith('_noise'):
                        current_event['Z_len'] = 'list'  # tidak perlu load semua
                    if prefix.endswith('.N') and not prefix.endswith('_noise'):
                        current_event['N_len'] = 'list'
                    if prefix.endswith('.E') and not prefix.endswith('_noise'):
                        current_event['E_len'] = 'list'
                    if prefix.endswith('.metadata.station'):
                        current_event['station'] = value
                    if prefix.endswith('.metadata.network'):
                        current_event['network'] = value
                    if prefix.endswith('.metadata.p_arrival'):
                        current_event['p_arrival'] = value
                    if prefix.endswith('.type'):
                        current_event['type'] = value
            
            # Hitung komponen di setiap event (untuk statistik)
            if event == 'map_key':
                if prefix.endswith('Z') and not prefix.endswith('_noise'):
                    keys_found.add('Z')
                if prefix.endswith('N') and not prefix.endswith('_noise'):
                    keys_found.add('N')
                if prefix.endswith('E') and not prefix.endswith('_noise'):
                    keys_found.add('E')
                if prefix.endswith('metadata'):
                    keys_found.add('metadata')
            
            # Simpan sampel setelah selesai membaca 3 event
            if total_entries == 3 and current_event and len(sample_entries) < 3:
                sample_entries.append(current_event.copy())
    
    # Tampilkan hasil
    print(f"\n✅ Total entri (event_id): {total_entries}")
    print(f"📋 Keys yang ditemukan di tiap entri: {sorted(keys_found)}")
    
    if sample_entries:
        print("\n📝 Contoh 3 event pertama (metadata):")
        for i, sample in enumerate(sample_entries, 1):
            print(f"  Event {i}:")
            print(f"    - station: {sample.get('station', 'N/A')}")
            print(f"    - network: {sample.get('network', 'N/A')}")
            print(f"    - p_arrival: {sample.get('p_arrival', 'N/A')}")
            print(f"    - type: {sample.get('type', 'N/A')}")
    
    # Cek validitas struktur
    print("\n" + "="*70)
    print("📊 VALIDASI STRUKTUR")
    print("="*70)
    
    if 'Z' in keys_found:
        print("✅ Komponen Z tersedia")
    else:
        print("❌ Komponen Z TIDAK DITEMUKAN (seharusnya wajib ada!)")
    
    if 'N' in keys_found:
        print("✅ Komponen N tersedia")
    else:
        print("⚠️ Komponen N tidak ditemukan (mungkin semua data 1C?)")
    
    if 'E' in keys_found:
        print("✅ Komponen E tersedia")
    else:
        print("⚠️ Komponen E tidak ditemukan (mungkin semua data 1C?)")
    
    if 'metadata' in keys_found:
        print("✅ Metadata tersedia (network, station, p_arrival, file)")
    else:
        print("❌ Metadata TIDAK DITEMUKAN!")
    
    print("\n" + "="*70)
    print("💡 CATATAN:")
    print("- File ini adalah JSON 3C, seharusnya berisi data Z, N, dan E.")
    print("- Jika hanya ada Z, mungkin karena source data hanya memiliki komponen Z.")
    print("- Untuk analisis lanjutan (cek panjang array 700), gunakan Skrip Opsi 2.")
    print("="*70)

if __name__ == "__main__":
    inspect_json_streaming(JSON_PATH)

🔍 MEMERIKSA JSON 3C (Streaming mode)
📂 File: /Volumes/Extreme SSD/json_juli_1c_3c/extracted_data_3c.json
📦 Ukuran file: 1.73 GB

⏳ Membaca file secara streaming... (mungkin butuh beberapa detik)

✅ Total entri (event_id): 16223
📋 Keys yang ditemukan di tiap entri: ['metadata']

📝 Contoh 3 event pertama (metadata):
  Event 1:
    - station: N/A
    - network: N/A
    - p_arrival: N/A
    - type: se
  Event 2:
    - station: N/A
    - network: N/A
    - p_arrival: N/A
    - type: se
  Event 3:
    - station: N/A
    - network: N/A
    - p_arrival: N/A
    - type: se

📊 VALIDASI STRUKTUR
❌ Komponen Z TIDAK DITEMUKAN (seharusnya wajib ada!)
⚠️ Komponen N tidak ditemukan (mungkin semua data 1C?)
⚠️ Komponen E tidak ditemukan (mungkin semua data 1C?)
✅ Metadata tersedia (network, station, p_arrival, file)

💡 CATATAN:
- File ini adalah JSON 3C, seharusnya berisi data Z, N, dan E.
- Jika hanya ada Z, mungkin karena source data hanya memiliki komponen Z.
- Untuk analisis lanjutan (cek panjang a

In [7]:
import ijson
import os

JSON_PATH = "/Volumes/Extreme SSD/json_juli_1c_3c/extracted_data_3c.json"

def inspect_structure_correctly(file_path):
    print("="*70)
    print("🔍 MEMERIKSA STRUKTUR JSON 3C (Metode yang lebih baik)")
    print("="*70)
    
    total_items = 0
    first_item_keys = None
    has_z = 0
    has_n = 0
    has_e = 0
    complete_3c = 0
    
    # Sample first 5 keys
    sample_keys = []
    
    with open(file_path, 'rb') as f:
        # Use kvitems to get top-level key-value pairs
        items = ijson.kvitems(f, '')
        
        for idx, (key, value) in enumerate(items):
            total_items += 1
            
            if idx == 0:
                first_item_keys = list(value.keys())
                # Check lengths
                z_len = len(value.get('Z', []))
                n_len = len(value.get('N', []))
                e_len = len(value.get('E', []))
                print(f"📋 Event pertama (key: {key}):")
                print(f"   Keys available: {first_item_keys}")
                print(f"   Length of Z: {z_len}")
                print(f"   Length of N: {n_len}")
                print(f"   Length of E: {e_len}")
                print(f"   Metadata: {value.get('metadata', {})}")
            
            # Count components for all items
            if 'Z' in value:
                has_z += 1
            if 'N' in value:
                has_n += 1
            if 'E' in value:
                has_e += 1
            if 'Z' in value and 'N' in value and 'E' in value:
                # verify lengths maybe
                if len(value.get('Z', [])) == 700 and len(value.get('N', [])) == 700 and len(value.get('E', [])) == 700:
                    complete_3c += 1
            
            if idx < 5:
                sample_keys.append(key)
            
            if idx >= 10000 and idx % 1000 == 0:
                print(f"   ...processed {idx} items")
    
    print(f"\n✅ Total entri: {total_items}")
    print(f"📋 Contoh key pertama: {sample_keys[:5]}")
    print(f"📊 Komponen Z ditemukan di: {has_z} entri")
    print(f"📊 Komponen N ditemukan di: {has_n} entri")
    print(f"📊 Komponen E ditemukan di: {has_e} entri")
    print(f"✅ Entri dengan Z,N,E lengkap (panjang 700): {complete_3c}")
    
    if has_z == total_items:
        print("✅ SEMUA entri memiliki komponen Z!")
    else:
        print(f"⚠️ {total_items - has_z} entri kehilangan Z.")

inspect_structure_correctly(JSON_PATH)

🔍 MEMERIKSA STRUKTUR JSON 3C (Metode yang lebih baik)
📋 Event pertama (key: GE_UGM_20100821_114140):
   Keys available: ['type', 'Z', 'N', 'E', 'Z_noise', 'N_noise', 'E_noise', 'metadata']
   Length of Z: 700
   Length of N: 700
   Length of E: 700
   Metadata: {'network': 'GE', 'station': 'UGM', 'p_arrival': '2010-08-21T11:41:41.119538Z', 'file': 'GE_UGM_20100821_114140.mseed'}
   ...processed 10000 items
   ...processed 11000 items
   ...processed 12000 items
   ...processed 13000 items
   ...processed 14000 items
   ...processed 15000 items
   ...processed 16000 items

✅ Total entri: 16223
📋 Contoh key pertama: ['GE_UGM_20100821_114140', 'GE_MMRI_20100903_084542', 'GE_SAUI_20100410_025324', 'GE_MNAI_20100324_092943', 'GE_TNTI_20100628_011841']
📊 Komponen Z ditemukan di: 16223 entri
📊 Komponen N ditemukan di: 16223 entri
📊 Komponen E ditemukan di: 16223 entri
✅ Entri dengan Z,N,E lengkap (panjang 700): 16223
✅ SEMUA entri memiliki komponen Z!


In [5]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
INSPEKSI JSON 3C - VERSI KOREKSI (Menggunakan ijson.kvitems)
"""

import ijson
import os
import json

JSON_PATH = "/Volumes/Extreme SSD/json_juli_1c_3c/extracted_data_3c.json"

def inspect_json_correctly(file_path):
    print("="*70)
    print("🔍 MEMERIKSA STRUKTUR JSON 3C (Metode kvitems)")
    print("="*70)
    print(f"📂 File: {file_path}")
    
    file_size = os.path.getsize(file_path) / (1024**3)
    print(f"📦 Ukuran file: {file_size:.2f} GB")
    
    total_entries = 0
    has_z = 0
    has_n = 0
    has_e = 0
    complete_3c = 0
    z_lengths = []
    first_event_info = None
    
    print("\n⏳ Membaca file secara streaming (satu per satu event)...")
    
    with open(file_path, 'rb') as f:
        # kvitems akan mengembalikan (key, value) untuk setiap top-level item
        # value adalah dictionary lengkap untuk satu event_id
        for idx, (event_id, event_data) in enumerate(ijson.kvitems(f, '')):
            total_entries += 1
            
            # Ambil data untuk event pertama saja (untuk sample)
            if idx == 0:
                keys_in_event = list(event_data.keys())
                metadata = event_data.get('metadata', {})
                z_arr = event_data.get('Z', [])
                n_arr = event_data.get('N', [])
                e_arr = event_data.get('E', [])
                
                first_event_info = {
                    'event_id': event_id,
                    'keys': keys_in_event,
                    'metadata': metadata,
                    'len_Z': len(z_arr),
                    'len_N': len(n_arr),
                    'len_E': len(e_arr),
                    'sample_Z_first_3': z_arr[:3] if len(z_arr) > 0 else []
                }
            
            # Hitung komponen yang tersedia
            if 'Z' in event_data:
                has_z += 1
                z_len = len(event_data['Z'])
                z_lengths.append(z_len)
            if 'N' in event_data:
                has_n += 1
            if 'E' in event_data:
                has_e += 1
            
            # Cek apakah lengkap 3C dan panjangnya 700
            if ('Z' in event_data and 'N' in event_data and 'E' in event_data and
                len(event_data['Z']) == 700 and 
                len(event_data['N']) == 700 and 
                len(event_data['E']) == 700):
                complete_3c += 1
            
            # Progress indicator (setiap 1000 entries)
            if total_entries % 1000 == 0:
                print(f"   Diproses {total_entries} entri...")
    
    # ============= TAMPILKAN HASIL =============
    print("\n" + "="*70)
    print("📊 HASIL INSPEKSI")
    print("="*70)
    
    print(f"✅ Total entri (event_id): {total_entries}")
    
    if first_event_info:
        print(f"\n📋 Contoh event pertama (ID: {first_event_info['event_id']}):")
        print(f"   - Keys yang tersedia: {first_event_info['keys']}")
        print(f"   - Metadata: {first_event_info['metadata']}")
        print(f"   - Panjang array Z: {first_event_info['len_Z']} (seharusnya 700)")
        print(f"   - Panjang array N: {first_event_info['len_N']} (seharusnya 700)")
        print(f"   - Panjang array E: {first_event_info['len_E']} (seharusnya 700)")
        if first_event_info['len_Z'] > 0:
            print(f"   - 3 nilai pertama Z: {first_event_info['sample_Z_first_3']}")
    
    print(f"\n📊 Statistik Komponen:")
    print(f"   - Memiliki Z: {has_z} entri ({has_z/total_entries*100:.2f}%)")
    print(f"   - Memiliki N: {has_n} entri ({has_n/total_entries*100:.2f}%)")
    print(f"   - Memiliki E: {has_e} entri ({has_e/total_entries*100:.2f}%)")
    print(f"   - Memiliki Z,N,E lengkap (panjang 700): {complete_3c} entri ({complete_3c/total_entries*100:.2f}%)")
    
    if z_lengths:
        unique_lengths = set(z_lengths)
        if len(unique_lengths) == 1:
            print(f"\n✅ Semua array Z memiliki panjang yang sama: {list(unique_lengths)[0]}")
        else:
            print(f"\n⚠️ Panjang array Z bervariasi: {sorted(unique_lengths)}")
    
    # Validasi
    print("\n" + "="*70)
    if complete_3c == total_entries:
        print("🎉 SEMUA DATA VALID: Semua entri memiliki Z, N, E dengan panjang 700.")
        print("✅ File JSON ini SIAP digunakan untuk inferensi MCU-Quake!")
    else:
        missing = total_entries - complete_3c
        print(f"⚠️ Ada {missing} entri yang tidak lengkap (kurang komponen atau panjang tidak 700).")
        print("   Jika Anda hanya butuh data 1C (Z saja), abaikan peringatan ini.")
        print("   Jika Anda butuh 3C, periksa kembali file .mseed sumber.")
    print("="*70)

if __name__ == "__main__":
    inspect_json_correctly(JSON_PATH)

🔍 MEMERIKSA STRUKTUR JSON 3C (Metode kvitems)
📂 File: /Volumes/Extreme SSD/json_juli_1c_3c/extracted_data_3c.json
📦 Ukuran file: 1.73 GB

⏳ Membaca file secara streaming (satu per satu event)...
   Diproses 1000 entri...
   Diproses 2000 entri...
   Diproses 3000 entri...
   Diproses 4000 entri...
   Diproses 5000 entri...
   Diproses 6000 entri...
   Diproses 7000 entri...
   Diproses 8000 entri...
   Diproses 9000 entri...
   Diproses 10000 entri...
   Diproses 11000 entri...
   Diproses 12000 entri...
   Diproses 13000 entri...
   Diproses 14000 entri...
   Diproses 15000 entri...
   Diproses 16000 entri...

📊 HASIL INSPEKSI
✅ Total entri (event_id): 16223

📋 Contoh event pertama (ID: GE_UGM_20100821_114140):
   - Keys yang tersedia: ['type', 'Z', 'N', 'E', 'Z_noise', 'N_noise', 'E_noise', 'metadata']
   - Metadata: {'network': 'GE', 'station': 'UGM', 'p_arrival': '2010-08-21T11:41:41.119538Z', 'file': 'GE_UGM_20100821_114140.mseed'}
   - Panjang array Z: 700 (seharusnya 700)
   - P